# INF4058 - Lab 3: NYC Yellow Taxi Data Cleaning

**Scope:** inspect 100 rows, profile the whole January 2021 CSV in **100,000-row chunks**, count data-quality issues, document decisions, implement `clean(df)`, verify the output, and **stop at cleaning**.

**Important:** Do **not** load cleaned data into PostgreSQL. This notebook does not connect to a database, and does not overwrite the downloaded data.

**Evidence:** take screenshots of the **printed** output tables (with code and output visible). Insert them into the Lab 3 PDF report. Class Docker screenshots must come from your real in-class execution.

## Step 0 - Imports and locate/download the source CSV.GZ

In [1]:
from pathlib import Path
from collections import Counter
import urllib.request
import pandas as pd
import numpy as np

CHUNK_SIZE = 100_000
DATA_NAME = "yellow_tripdata_2021-01.csv.gz"
DATA_URL = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz"
DATA_PATH = Path(DATA_NAME)

# If you have the CSV.GZ next to this notebook, the download will be skipped.
if not DATA_PATH.is_file():
    print(f"Downloading {DATA_NAME} ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

assert DATA_PATH.is_file(), f"File not found: {DATA_PATH}"
print("Data file:", DATA_PATH.resolve())
print("Compressed file size (MB):", round(DATA_PATH.stat().st_size / (1024**2), 2))
print("Chunk size:", CHUNK_SIZE)


Data file: C:\Users\BIOS\Downloads\yellow_tripdata_2021-01.csv.gz
Compressed file size (MB): 23.87
Chunk size: 100000


## Step 1 - Look at the first 100 rows

Each row represents **one recorded yellow taxi trip**, with pickup and drop-off times, locations, distance, passengers, and payment amounts. The date columns are `tpep_pickup_datetime` and `tpep_dropoff_datetime`. Nullable integer columns use Pandas `Int64` (capital **I**) so missing IDs remain missing rather than being converted into arbitrary floats.

In [2]:
DATE_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
INT_COLS = ["VendorID", "passenger_count", "RatecodeID", "PULocationID", "DOLocationID", "payment_type"]
DTYPE_MAP = {c: "Int64" for c in INT_COLS}
DTYPE_MAP["store_and_fwd_flag"] = "string"

def read_chunks(path=DATA_PATH):
    """Yield 100,000-row batches with consistent analysis dtypes. No rows are changed or dropped."""
    for part in pd.read_csv(path, chunksize=CHUNK_SIZE, dtype=DTYPE_MAP, low_memory=False):
        for col in DATE_COLS:
            part[col] = pd.to_datetime(part[col], errors="coerce")
        yield part

first_100 = pd.read_csv(DATA_PATH, nrows=100, dtype=DTYPE_MAP)
for date_col in DATE_COLS:
    first_100[date_col] = pd.to_datetime(first_100[date_col], errors="coerce")

with pd.option_context("display.max_columns", None, "display.width", 240):
    print("=== df.head() ===")
    print(first_100.head())
    print("\n=== df.dtypes ===")
    print(first_100.dtypes)
    print("\n=== NULL COUNTS (first 100 rows) ===")
    print(first_100.isna().sum())
print("\nOne row = one taxi trip; date columns =", DATE_COLS)


=== df.head() ===
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5        0.00           0.0                    0.3   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5        0.00           0.0                    0.3   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132           165             1         42.0    0.5      0.5        8.65           0.0                    0.3   
3         1  2021-

## Step 2 - Full-data profile + Step 3 - Count the problems

Read the entire file in **100,000-row chunks**. The first pass is **read-only**: we collect statistics but do **not** drop, fix, or flag any source rows. Date parsing is only a type interpretation for analysis.

Duplicate counting is **global across chunks**, based on a stable 64-bit Pandas hash of each complete row. The hashing method has a negligible but nonzero theoretical collision risk; in normal class-sized datasets, it is practical. The source file is unchanged.

In [3]:
JAN_START = pd.Timestamp("2021-01-01")
FEB_START = pd.Timestamp("2021-02-01")

problem_counts = Counter()
null_totals = Counter()
column_dtypes = {}
minimums, maximums = {}, {}
profile_seen_hashes = set()
total_rows = 0
source_cols = None

def update_range(column, series):
    valid = series.dropna()
    if valid.empty:
        return
    this_min, this_max = valid.min(), valid.max()
    if column not in minimums or this_min < minimums[column]:
        minimums[column] = this_min
    if column not in maximums or this_max > maximums[column]:
        maximums[column] = this_max

for batch_num, chunk in enumerate(read_chunks(), start=1):
    if source_cols is None:
        source_cols = list(chunk.columns)
    total_rows += len(chunk)

    # Profile all columns (no cleaning in this pass).
    for col in source_cols:
        column_dtypes[col] = str(chunk[col].dtype)
        null_totals[col] += int(chunk[col].isna().sum())
        update_range(col, chunk[col])

    pickup, dropoff = chunk[DATE_COLS[0]], chunk[DATE_COLS[1]]
    outside = pickup.notna() & ~pickup.between(JAN_START, FEB_START, inclusive="left")
    problem_counts["missing_or_invalid_pickup_date"] += int(pickup.isna().sum())
    problem_counts["missing_or_invalid_dropoff_date"] += int(dropoff.isna().sum())
    problem_counts["pickup_outside_jan_2021"] += int(outside.sum())
    problem_counts["negative_fare"] += int((chunk["fare_amount"] < 0).sum())
    problem_counts["zero_fare"] += int((chunk["fare_amount"] == 0).sum())
    problem_counts["negative_distance"] += int((chunk["trip_distance"] < 0).sum())
    problem_counts["zero_distance"] += int((chunk["trip_distance"] == 0).sum())
    problem_counts["zero_passengers"] += int((chunk["passenger_count"] == 0).sum())

    # Compare full rows, including duplicates from earlier chunks.
    row_hashes = pd.util.hash_pandas_object(chunk[source_cols], index=False)
    for h in row_hashes.to_numpy(dtype="uint64"):
        key = int(h)
        if key in profile_seen_hashes:
            problem_counts["duplicate_full_rows"] += 1
        else:
            profile_seen_hashes.add(key)
    print(f"Profiled chunk {batch_num:02d}: {len(chunk):,} rows | cumulative: {total_rows:,}")

profile_table = pd.DataFrame([
    {
        "column": col,
        "dtype": column_dtypes[col],
        "null_count": null_totals[col],
        "null_pct": round(100.0 * null_totals[col] / total_rows, 3),
        "min": str(minimums.get(col, "NA")),
        "max": str(maximums.get(col, "NA")),
    }
    for col in source_cols
])

problem_labels = [
    "missing_or_invalid_pickup_date", "missing_or_invalid_dropoff_date",
    "pickup_outside_jan_2021", "negative_fare", "zero_fare",
    "negative_distance", "zero_distance", "zero_passengers", "duplicate_full_rows"
]
problems_table = pd.DataFrame([
    {"problem": k, "count": problem_counts[k]} for k in problem_labels
])

print("\n=== FULL-DATA PROFILE TABLE (actual computed values) ===")
with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.width", 240, "display.max_colwidth", 40):
    print(profile_table)
print("\n=== PROBLEM COUNTS (actual computed values) ===")
print(problems_table)
print("\nTotal source rows:", total_rows)


Profiled chunk 01: 100,000 rows | cumulative: 100,000
Profiled chunk 02: 100,000 rows | cumulative: 200,000
Profiled chunk 03: 100,000 rows | cumulative: 300,000
Profiled chunk 04: 100,000 rows | cumulative: 400,000
Profiled chunk 05: 100,000 rows | cumulative: 500,000
Profiled chunk 06: 100,000 rows | cumulative: 600,000
Profiled chunk 07: 100,000 rows | cumulative: 700,000
Profiled chunk 08: 100,000 rows | cumulative: 800,000
Profiled chunk 09: 100,000 rows | cumulative: 900,000
Profiled chunk 10: 100,000 rows | cumulative: 1,000,000
Profiled chunk 11: 100,000 rows | cumulative: 1,100,000
Profiled chunk 12: 100,000 rows | cumulative: 1,200,000
Profiled chunk 13: 100,000 rows | cumulative: 1,300,000
Profiled chunk 14: 69,765 rows | cumulative: 1,369,765

=== FULL-DATA PROFILE TABLE (actual computed values) ===
                   column           dtype  null_count  null_pct                  min                  max
0                VendorID           Int64       98352      7.18        

## Step 4 - Decide what to drop or flag

The table documents the **decision, reason, and rule owner**. Some conditions can overlap, so initial problem counts are **not** added together. During cleaning, removal rules are applied **sequentially** to the rows still remaining, which gives auditable, non-overlapping removal counts.

**Business interpretation:** A zero fare or a zero-distance trip can sometimes be legitimate (e.g., a canceled/no-charge ride), and an apparent zero passenger count can be a reporting issue. We preserve these as **flags** until a business/data steward approves a stronger rule.

In [4]:
decisions_table = pd.DataFrame([
    ["Missing/invalid pickup or dropoff date", "DROP", "Cannot reliably locate the trip in time", "Data Engineering / Data Steward"],
    ["Pickup outside January 2021", "DROP", "This cleaning batch represents Jan 2021 trips", "Data Engineering / Data Steward"],
    ["Negative fare_amount", "DROP", "Invalid amount for this simplified trip-cleaning rule", "Finance / Data Steward"],
    ["Negative trip_distance", "DROP", "A trip cannot have negative traveled distance", "Operations / Data Steward"],
    ["Exact duplicate full row", "DROP", "Avoid double-counting identical trips", "Data Engineering"],
    ["Zero fare_amount", "FLAG", "Could be a no-charge or exceptional trip", "Finance / Data Steward"],
    ["Zero trip_distance", "FLAG", "Could be a canceled or exceptional trip", "Operations / Data Steward"],
    ["passenger_count == 0", "FLAG", "Needs clarification from trip-data owner", "Operations / Data Steward"],
    ["Missing fare/distance/passenger count", "FLAG", "Preserve ambiguous values for data-owner review", "Operations / Data Steward"],
], columns=["problem", "decision", "why", "owner"])
with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.width", 240, "display.max_colwidth", 55):
    print(decisions_table)


                                  problem decision                                                    why                            owner
0  Missing/invalid pickup or dropoff date     DROP                Cannot reliably locate the trip in time  Data Engineering / Data Steward
1             Pickup outside January 2021     DROP          This cleaning batch represents Jan 2021 trips  Data Engineering / Data Steward
2                    Negative fare_amount     DROP  Invalid amount for this simplified trip-cleaning rule           Finance / Data Steward
3                  Negative trip_distance     DROP          A trip cannot have negative traveled distance        Operations / Data Steward
4                Exact duplicate full row     DROP                  Avoid double-counting identical trips                 Data Engineering
5                        Zero fare_amount     FLAG               Could be a no-charge or exceptional trip           Finance / Data Steward
6                      Zero

## Step 5 - Implement and run `clean(df)`

`clean(df)` returns exactly **(cleaned_dataframe, rows_removed_by_rule)** and does not change its input. The function tracks exact-duplicate hashes across batches via `seen_clean_hashes`, which is initialized before the full cleaning run. The new `flag_*` columns document odd records **without removing** them.

In [5]:
seen_clean_hashes = set()
REMOVAL_RULES = [
    "invalid_date", "pickup_outside_jan_2021", "negative_fare",
    "negative_distance", "duplicate_full_rows"
]
FLAG_COLS = [
    "flag_zero_fare", "flag_zero_distance",
    "flag_zero_passengers", "flag_missing_fare_distance_or_passengers"
]

def clean(df):
    """Clean one batch; keep the input intact; return (cleaned_df, {rule: rows_removed}).

    Expects consistent date dtypes from read_chunks(). `seen_clean_hashes` is the
    intentionally shared state used to detect duplicate rows across batches.
    """
    result = df.copy(deep=True)
    rows_removed = {r: 0 for r in REMOVAL_RULES}

    def drop_rows(rule, bad_mask):
        nonlocal result
        bad_mask = bad_mask.fillna(False)
        rows_removed[rule] = int(bad_mask.sum())
        result = result.loc[~bad_mask].copy()

    # Interpret date columns, without touching the original df.
    for col in DATE_COLS:
        result[col] = pd.to_datetime(result[col], errors="coerce")

    drop_rows("invalid_date", result[DATE_COLS].isna().any(axis=1))
    out_of_month = ~result[DATE_COLS[0]].between(JAN_START, FEB_START, inclusive="left")
    drop_rows("pickup_outside_jan_2021", out_of_month)
    drop_rows("negative_fare", result["fare_amount"] < 0)
    drop_rows("negative_distance", result["trip_distance"] < 0)

    # Deduplicate ORIGINAL source columns only, before adding flag columns.
    hashes = pd.util.hash_pandas_object(result[source_cols], index=False)
    duplicate_mask = []
    for h in hashes.to_numpy(dtype="uint64"):
        key = int(h)
        duplicate = key in seen_clean_hashes
        duplicate_mask.append(duplicate)
        if not duplicate:
            seen_clean_hashes.add(key)
    drop_rows("duplicate_full_rows", pd.Series(duplicate_mask, index=result.index))

    result["flag_zero_fare"] = result["fare_amount"].eq(0).fillna(False).astype(bool)
    result["flag_zero_distance"] = result["trip_distance"].eq(0).fillna(False).astype(bool)
    result["flag_zero_passengers"] = result["passenger_count"].eq(0).fillna(False).astype(bool)
    result["flag_missing_fare_distance_or_passengers"] = result[
        ["fare_amount", "trip_distance", "passenger_count"]
    ].isna().any(axis=1)

    assert sum(rows_removed.values()) == len(df) - len(result)
    return result, rows_removed

before_count = 0
after_count = 0
removal_totals = Counter({r: 0 for r in REMOVAL_RULES})
flag_totals = Counter({col: 0 for col in FLAG_COLS})
verification_counts = Counter()

for batch_num, chunk in enumerate(read_chunks(), start=1):
    original_snapshot = chunk.copy(deep=True)
    cleaned, rules = clean(chunk)
    assert chunk.equals(original_snapshot), "The input DataFrame was modified."
    before_count += len(chunk)
    after_count += len(cleaned)
    removal_totals.update(rules)
    flag_totals.update({col: int(cleaned[col].sum()) for col in FLAG_COLS})

    # The conditions chosen for dropping must now be absent.
    verification_counts["remaining_invalid_dates"] += int(cleaned[DATE_COLS].isna().any(axis=1).sum())
    verification_counts["remaining_non_january_pickups"] += int((~cleaned[DATE_COLS[0]].between(JAN_START, FEB_START, inclusive="left")).sum())
    verification_counts["remaining_negative_fares"] += int((cleaned["fare_amount"] < 0).sum())
    verification_counts["remaining_negative_distances"] += int((cleaned["trip_distance"] < 0).sum())
    print(f"Cleaned chunk {batch_num:02d}: {len(chunk):,} -> {len(cleaned):,} rows")

removal_table = pd.DataFrame([{"rule": rule, "rows_removed": removal_totals[rule]} for rule in REMOVAL_RULES])
flags_table = pd.DataFrame([{"flag": flag, "rows_flagged_after_cleaning": flag_totals[flag]} for flag in FLAG_COLS])
print("\n=== CLEANING SUMMARY ===")
print(pd.DataFrame([{
    "rows_before": before_count,
    "rows_after": after_count,
    "rows_removed_total": before_count - after_count,
}]))
print("\n=== ROWS REMOVED PER RULE ===")
print(removal_table)
print("\n=== ROWS FLAGGED (NOT REMOVED) ===")
print(flags_table)
print("\n=== RULE: ROWS_REMOVED DICTIONARY ===")
print(dict(removal_totals))


Cleaned chunk 01: 100,000 -> 99,329 rows
Cleaned chunk 02: 100,000 -> 99,480 rows
Cleaned chunk 03: 100,000 -> 99,479 rows
Cleaned chunk 04: 100,000 -> 99,479 rows
Cleaned chunk 05: 100,000 -> 99,507 rows
Cleaned chunk 06: 100,000 -> 99,499 rows
Cleaned chunk 07: 100,000 -> 99,493 rows
Cleaned chunk 08: 100,000 -> 99,495 rows
Cleaned chunk 09: 100,000 -> 99,497 rows
Cleaned chunk 10: 100,000 -> 99,470 rows
Cleaned chunk 11: 100,000 -> 99,483 rows
Cleaned chunk 12: 100,000 -> 99,485 rows
Cleaned chunk 13: 100,000 -> 99,591 rows
Cleaned chunk 14: 69,765 -> 69,685 rows

=== CLEANING SUMMARY ===
   rows_before  rows_after  rows_removed_total
0      1369765     1362972                6793

=== ROWS REMOVED PER RULE ===
                      rule  rows_removed
0             invalid_date             0
1  pickup_outside_jan_2021            24
2            negative_fare          6769
3        negative_distance             0
4      duplicate_full_rows             0

=== ROWS FLAGGED (NOT REMOVED

## Step 6 - Verify and **stop**

No PostgreSQL write happens here. `before == after + removed`; all four invalid conditions targeted by dropping must be zero in the cleaned output. Because we kept only per-chunk cleaned DataFrames and counts, the original source file remains untouched and no data files need to be pushed to GitHub.

In [6]:
assert before_count == total_rows, "Profiling and cleaning row totals disagree."
assert before_count == after_count + sum(removal_totals.values())
assert all(value == 0 for value in verification_counts.values()), verification_counts
assert after_count == len(seen_clean_hashes), "Duplicate rows remain across cleaned chunks."

print("=== FINAL VERIFICATION ===")
print(pd.DataFrame([
    ["Profile and clean scanned same source rows", before_count == total_rows],
    ["Input rows = kept rows + removed rows", before_count == after_count + sum(removal_totals.values())],
    ["No invalid dates / out-of-month / negative values", all(v == 0 for v in verification_counts.values())],
    ["No duplicate complete rows across cleaned chunks", after_count == len(seen_clean_hashes)],
    ["No PostgreSQL load in this notebook", True],
], columns=["check", "passed"]))
print("\nRows before:", before_count)
print("Rows after:", after_count)
print("Rows removed:", before_count - after_count)
print("\nSTOP: CLEANING COMPLETE. Do NOT load data into PostgreSQL for Lab 3.")


=== FINAL VERIFICATION ===
                                               check  passed
0         Profile and clean scanned same source rows    True
1              Input rows = kept rows + removed rows    True
2  No invalid dates / out-of-month / negative values    True
3   No duplicate complete rows across cleaned chunks    True
4                No PostgreSQL load in this notebook    True

Rows before: 1369765
Rows after: 1362972
Rows removed: 6793

STOP: CLEANING COMPLETE. Do NOT load data into PostgreSQL for Lab 3.


## Step 7 - PDF report and GitHub checklist

The PDF must contain **seven sections**: **Cover** (name, ID, date, repo URL); **The flow in your words** (Source -> Extract -> Transform -> Load -> Serve and explanation of the class scripts); **What we ran in class** (your actual Compose, ingest, `COUNT = 1,369,765`, build and run screenshots); **What you found in the data** (screenshots of `profile_table` and `problems_table`); **How you cleaned it** (decisions, rule removals, before/after); **Answers** (three questions); **What broke** (your genuine error, cause, fix and screenshots).

**Three questions - wording to adapt to your own understanding:**

- **Q1:** `pgdatabase` is the PostgreSQL service/container DNS name on the shared Docker network. `localhost` inside an ingestion container means **that ingestion container itself**, not the database.
- **Q2:** Chunking bounds memory while reading large CSV files. Replacing the table for every month would delete data already loaded; `replace` is used once for a fresh table, then `append` for remaining batches/months (in class, not in Lab 3 cleaning).
- **Q3:** We printed full profile and problem counts, reconciled `rows_before = rows_after + rows_removed`, verified the input DataFrame was unchanged, checked no excluded conditions or duplicate complete rows remain, and did not load the cleaned data into PostgreSQL.

**GitHub:** Push the `.ipynb` (or a `.py` script) only. Add a `.gitignore` excluding `*.csv`, `*.csv.gz`, `.venv/`, `.env`, `__pycache__/`, and credentials. **Do not manufacture in-class screenshots**; use the ones you actually took.
